# 5. Create core nodes

Purpose: build and inspect canonical core KG node Parquets from OpenTargets/source snapshots using existing `manage_db` ingestion functions.

Prerequisites:
- Notebook 4 has downloaded or verified raw source snapshots for `target`, `disease`, `drug_molecule`, `biosample`, `go`, and `reactome` as needed.
- A writable staging KG root for full node builds; default `data/kg`.

Outputs:
- Node Parquets under `{KG_ROOT}/nodes/`, especially `gene`, `protein`, `transcript`, `disease`, `molecule`, `pathway`, `tissue`, `cell_type`, `cell_line`, `organism`, `phenotype`, `mutation`, `enhancer`, and `dataset` when their source steps apply.
- Count/schema/xref coverage tables.

Expected runtime:
- Default dry run: seconds; imports schema and inspects existing/local node files only.
- Full node build: minutes to longer depending on raw source size.

Safe sample mode:
- `DRY_RUN=True` by default; full writes require `JOUVENCE_NOTEBOOK_ALLOW_WRITES=1` and `JOUVENCE_NOTEBOOK_SAMPLE_MODE=0` (legacy fallbacks: `TXGNN_NOTEBOOK_ALLOW_WRITES` and `TXGNN_NOTEBOOK_SAMPLE_MODE`).
- Build cells call existing `manage_db.ingest_opentargets` functions. Inline code is limited to orchestration and validation display.

Biomedical KG policy reminders:
- Source-native IDs first.
- Gene-level OpenTargets rows create gene nodes/relations; do not project gene/RNA evidence into protein relations.
- Protein/transcript/enhancer/mutation nodes are only created when source-native assertions support them.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    # Allows running the notebook from notebooks/ in Jupyter.
    candidate = REPO_ROOT.parent
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
ALLOW_NETWORK = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_NETWORK", os.environ.get("TXGNN_NOTEBOOK_ALLOW_NETWORK", "0")) == "1"
ALLOW_WRITES = os.environ.get("JOUVENCE_NOTEBOOK_ALLOW_WRITES", os.environ.get("TXGNN_NOTEBOOK_ALLOW_WRITES", "0")) == "1"
OPEN_TARGETS_RELEASE = os.environ.get("OPEN_TARGETS_RELEASE", "latest")
DEFAULT_LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))
LOCAL_CACHE_ROOT = Path(os.environ.get("JOUVENCE_LOCAL_CACHE_ROOT", os.environ.get("TXGNN_LOCAL_CACHE_ROOT", DEFAULT_LOCAL_CACHE_ROOT)))
DATA_ROOT = Path(os.environ.get("JOUVENCE_DATA_ROOT", os.environ.get("TXGNN_DATA_ROOT", REPO_ROOT / "data")))
OT_ROOT = Path(os.environ.get("JOUVENCE_OT_ROOT", os.environ.get("TXGNN_OT_ROOT", DATA_ROOT / "opentargets")))
KG_ROOT_URI = os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(DATA_ROOT / "kg")))
CANONICAL_GCS_ROOT = "gs://jouvencekb/kg/v2"
CANONICAL_MOUNT_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "sample_mode": SAMPLE_MODE,
    "allow_network": ALLOW_NETWORK,
    "allow_writes": ALLOW_WRITES,
    "open_targets_release": OPEN_TARGETS_RELEASE,
    "local_cache_root": str(LOCAL_CACHE_ROOT),
    "ot_root": str(OT_ROOT),
    "kg_root_uri": KG_ROOT_URI,
}, indent=2))

DRY_RUN = SAMPLE_MODE or not ALLOW_WRITES
print({"dry_run": DRY_RUN})


## Import existing builders and schema helpers

The notebook delegates node creation to `manage_db.ingest_opentargets` and storage writes to `manage_db.kg_storage`.


In [ ]:
from manage_db import ingest_opentargets
from manage_db.ingest_opentargets import (
    ingest_targets,
    ingest_diseases,
    ingest_drugs,
    ingest_biosample,
    ingest_go,
    ingest_variants,
    ingest_enhancers,
)
from manage_db.kg_storage import open_kg_root
from manage_db.kg_schema import NODE_TYPES, NodeType, xref_columns_for, primary_ontology_for

class DryKGRoot:
    def __init__(self, uri: str):
        self.uri = uri.rstrip("/")
    def node_path(self, node_type: str) -> str:
        return f"{self.uri}/nodes/{node_type}.parquet"

CORE_NODE_DATASETS = ["target", "disease", "drug_molecule", "biosample", "go", "reactome"]
OPTIONAL_NODE_DATASETS = ["variant", "enhancer_to_gene", "disease_phenotype"]
CORE_BUILD_ORDER = [
    ("target", ingest_targets),
    ("disease", ingest_diseases),
    ("drug_molecule", ingest_drugs),
    ("biosample", ingest_biosample),
    ("go", ingest_go),
]
if DRY_RUN and "://" not in KG_ROOT_URI and not Path(KG_ROOT_URI).exists():
    kg_root = DryKGRoot(KG_ROOT_URI)
else:
    kg_root = open_kg_root(KG_ROOT_URI)
print({"kg_root": kg_root.uri, "ot_root": str(OT_ROOT), "core_build_order": [name for name, _ in CORE_BUILD_ORDER]})


## Node schema source of truth

This table is generated from `manage_db.kg_schema`, not hand-maintained in the notebook.


In [ ]:
schema_rows = []
for nt, info in NODE_TYPES.items():
    schema_rows.append({
        "node_type": nt.value,
        "primary_ontology": info.primary_ontology,
        "xref_columns": list(info.xref_columns),
    })
schema_table = pd.DataFrame(schema_rows).sort_values("node_type")
schema_table


## Source availability

This is the cheap gate before any build. Missing source directories are reported rather than silently ignored.


In [ ]:
def parquet_files(path: Path) -> list[Path]:
    return sorted(path.glob("*.parquet")) if path.exists() else []


def first_parquet_schema(path: Path) -> dict:
    files = parquet_files(path) if path.is_dir() else ([path] if path.exists() else [])
    if not files:
        return {"exists": False, "path": str(path), "parquet_files": 0}
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(files[0])
    return {
        "exists": True,
        "path": str(files[0]),
        "parquet_files": len(files),
        "rows_in_first_file": pf.metadata.num_rows,
        "columns": pf.schema_arrow.names,
    }


def duckdb_count(path: Path) -> int | None:
    if not path.exists():
        return None
    import duckdb
    target = str(path / "*.parquet") if path.is_dir() else str(path)
    with duckdb.connect() as con:
        return con.execute("select count(*) from read_parquet(?)", [target]).fetchone()[0]

source_status = []
for dataset in CORE_NODE_DATASETS + OPTIONAL_NODE_DATASETS:
    source_path = OT_ROOT / dataset
    schema = first_parquet_schema(source_path)
    source_status.append({
        "dataset": dataset,
        "path": str(source_path),
        "exists": source_path.exists(),
        "parquet_files": schema.get("parquet_files", 0),
        "sample_columns": schema.get("columns", [])[:16],
    })
source_status = pd.DataFrame(source_status)
source_status


## Build core nodes (full run only)

The default path is dry-run. To perform writes, set `JOUVENCE_NOTEBOOK_SAMPLE_MODE=0` and `JOUVENCE_NOTEBOOK_ALLOW_WRITES=1`, with `JOUVENCE_KG_ROOT` pointing to a staging KG root, not the canonical production root unless you are explicitly promoting. The corresponding `TXGNN_NOTEBOOK_SAMPLE_MODE`, `TXGNN_NOTEBOOK_ALLOW_WRITES`, and `TXGNN_KG_ROOT` names remain legacy fallbacks.


In [ ]:
results = {}
if DRY_RUN:
    print("DRY_RUN: not building nodes. Full-run equivalent:")
    print(
        "uv run python -m manage_db.ingest_opentargets "
        f"--data-dir {str(DATA_ROOT)!r} --release {OPEN_TARGETS_RELEASE} "
        "--datasets target disease drug_molecule biosample go --no-download"
    )
else:
    for dataset, fn in CORE_BUILD_ORDER:
        if not (OT_ROOT / dataset).exists():
            results[dataset] = "MISSING_SOURCE"
            continue
        results[dataset] = fn(OT_ROOT, DATA_ROOT / "kg", kg_root)
results


## Node output paths and count/schema checks

Counts use Parquet metadata where possible. Xref checks compare columns to `NODE_TYPES` so schema drift is caught early.


In [ ]:
def inspect_node(node_type: str) -> dict:
    public_path = kg_root.node_path(node_type)
    local_path = Path(public_path) if "://" not in public_path else None
    row = {"node_type": node_type, "path": public_path, "exists": False, "rows": None, "columns": [], "missing_xref_columns": []}
    if local_path is None or not local_path.exists():
        return row
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(local_path)
    cols = pf.schema_arrow.names
    expected_xrefs = list(xref_columns_for(NodeType(node_type)))
    row.update({
        "exists": True,
        "rows": pf.metadata.num_rows,
        "columns": cols,
        "missing_xref_columns": [c for c in expected_xrefs if c not in cols],
    })
    return row

node_checks = pd.DataFrame([inspect_node(nt.value) for nt in sorted(NODE_TYPES.keys(), key=lambda n: n.value)])
node_checks


## Namespace spot checks

These are lightweight examples of source-native endpoint policy. They only run against node files that exist locally.


In [ ]:
namespace_expectations = {
    "gene": ("ENSG",),
    "transcript": ("ENST",),
    "protein": ("ENSP", "UniProt"),
    "disease": ("EFO", "MONDO", "HP", "Orphanet"),
    "molecule": ("CHEMBL",),
    "pathway": ("GO", "Reactome", "REACTOME"),
    "tissue": ("UBERON",),
    "cell_type": ("CL",),
    "organism": ("NCBITaxon", "NCBITAXON"),
    "phenotype": ("HP", "PATO", "MP"),
    "mutation": ("rs", "chr", "COSV", "VAR"),
    "enhancer": ("EH", "ENSR", "chr"),
}

spot_rows = []
for node_type, prefixes in namespace_expectations.items():
    public_path = kg_root.node_path(node_type)
    local_path = Path(public_path) if "://" not in public_path else None
    if local_path is None or not local_path.exists():
        spot_rows.append({"node_type": node_type, "checked": False, "reason": "missing local node parquet", "expected_prefixes": prefixes})
        continue
    import pyarrow.parquet as pq
    table = pq.read_table(local_path, columns=["id"]).slice(0, 1000).to_pandas()
    ids = table["id"].dropna().astype(str)
    matched = bool(ids.empty) or ids.str.startswith(prefixes).any()
    spot_rows.append({"node_type": node_type, "checked": True, "expected_prefixes": prefixes, "sample_match": matched, "sample_ids": ids.head(5).tolist()})
spot_checks = pd.DataFrame(spot_rows)
spot_checks


## Validation

In dry run, verify imports/schema and ensure any existing node Parquets have required xref columns. In full run, require core node outputs to exist.


In [ ]:
assert set(["gene", "disease", "molecule"]).issubset({nt.value for nt in NODE_TYPES}), "core node schema missing expected node types"
existing_with_missing_xrefs = node_checks[node_checks["exists"] & node_checks["missing_xref_columns"].map(bool)]
assert existing_with_missing_xrefs.empty, existing_with_missing_xrefs[["node_type", "missing_xref_columns"]].to_dict(orient="records")
if not DRY_RUN:
    required_core_outputs = ["gene", "disease", "molecule"]
    missing_outputs = node_checks[node_checks["node_type"].isin(required_core_outputs) & ~node_checks["exists"]]["node_type"].tolist()
    assert not missing_outputs, f"Full run expected core node files: {missing_outputs}"
print("Notebook 5 lightweight validation passed.")
